# O+O 200 GeV bulk against STAR: multiplicity

Produced by `oo_bulk.py` (branch `oo_rhic_run`) from the events of the centrality-bias test (`oo_bias.py`,
notebook `oo_centrality_bias.ipynb`); no events of its own. Model results, not vetted physics results.

## Purpose
The O+O runs take the bulk set of `config/OO_MCGlauber_MUSIC_isr_rhictune.xml` (3dMCGlauber + MUSIC + iSS,
the RHIC bulk tune of arXiv:2601.17234, EOS 91) unchanged; only the hard sector comes from the p+p tune. The
plan validates that bulk set against STAR's O+O data instead of retuning it. Multiplicity matters twice:
it is the bulk observable the tune should get right, and it is what the experiments sort events by, so the
centrality-bias result (R4b) depends on the model's multiplicity distribution in the estimator's acceptance.

## Data (STAR, O+O 200 GeV; no HEPData for these figures)
Read exactly from the vector figures in the arXiv sources (`refs/star_oo/extract_star_oo.py` →
`refs/star_oo/star_oo_bulk.json`):
- **arXiv:2510.19645, Fig. 5** (flow paper): p($N_\mathrm{ch}$) and $\langle N_\mathrm{ch}\rangle$ vs centrality;
  $N_\mathrm{ch}$ corrected for tracking efficiency, $0.2<p_T<2$ GeV, $|\eta|<1.5$; centrality from a
  Glauber + negative-binomial fit to p($N_\mathrm{ch}$) (which also corrects the trigger inefficiency at low $N_\mathrm{ch}$).
- **arXiv:2604.13935, Figs. 1 and 4** (jet quenching paper): the raw number of MIPs in the EPD
  ($2.14<|\eta|<5.09$) for minimum-bias events (trigger-efficiency corrected) and for events with a charged
  trigger at $|\eta|<1.5$ (5–7, 7–10, 10–30 GeV), self-normalised; the event-activity class edges; Fig. 4 the
  ratios to the 5–7 GeV distribution.

## Model
- **Minimum bias**: the `mb` sample of `oo_bias.py` — the ISR chain without the hard-momentum subtraction and
  with the hard-pair string veto off, only the bulk hadrons counted. Same strings and geometry as minimum-bias
  3dMCGlauber events (closure in `closure_oo.ipynb`, part 1d), $b$ 0–10 fm, MUSIC + iSS with one sample per
  event (the physical fluctuations), charged hadrons, iSS decays (η, Σ⁰ stable; weak decays done).
- **Jet events**: the `jets` sample (production settings: subtraction on, veto on; jet in vacuum), $N_\mathrm{coll}$
  weighted, $\hat p_T$ windows 9–11, 18–23, 30–40, 40–55 GeV; bulk + the hard event's hadrons.
- **Model centrality** for Fig. 5: percentiles of the model's own $N_\mathrm{ch}$ over all events (every event has
  at least one NN collision, i.e. the full inelastic cross section, as STAR's fit).
- **EPD**: nMIP is not $N_\mathrm{ch}$ (secondaries from material, Landau fluctuations, tile thresholds); the model's
  charged multiplicity in the EPD acceptance is mapped with one scale $k$ (MIPs per primary charged particle),
  fitted to STAR's minimum-bias shape (bins above nMIP 78), optionally with a Gaussian response width
  $s\sqrt{N}$ per event. $k$ is a free nuisance parameter, so only the **shape** is tested; the class edges give
  $k$ edge by edge — a constant $k$ means the shape is right.
- **Errors**: every model point or curve carries its statistical error (bars or ±1σ bands; `hstat.py`): one hydro
  and one iSS sample per event, so the events are independent; particle distributions (dN/dη) use the
  event-to-event spread. STAR's points are read from the figures; their statistical errors are smaller than the
  markers and are not drawn.

## Known differences to the bulk-tune paper
Freeze-out at $T=0.15$ GeV instead of the paper's $e_\mathrm{sw}=0.52$ GeV/fm³; no transverse string shift
($\alpha_\mathrm{shift}=0$); no UrQMD (iSS decays); η not decayed. These move the overall multiplicity by a few
percent and barely the shape.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import tune as T
import hstat as H   # statistical errors: one hydro + iSS sample per event (no oversampling), errors from the events
R = json.loads((T.RES / "oo_bulk.json").read_text())
S = json.loads((T.RES.parent / "refs" / "star_oo" / "star_oo_bulk.json").read_text())
print(f"model minimum bias: {R['n_mb']} events; <N_ch>(|eta|<1.5, 0.2<pT<2) {R['pnch']['mean']:.1f}; "
      f"<N_ch> in the EPD {R['epd']['mean_n_epd']:.1f}")
print("jet events:", R["jets"]["n"] if "jets" in R else "not yet")

## 1. p($N_\mathrm{ch}$) in STAR's flow acceptance
STAR's distribution has an arbitrary scale and loses events below $N_\mathrm{ch}\approx 10$ (trigger), so both are
normalised to unit area above $N_\mathrm{ch}=15$. Left: as is; right: the model's $N_\mathrm{ch}$ scaled by the
factor $\alpha$ that best matches the shape above 15 — $\alpha$ is the overall multiplicity deficit, the residual
is the shape.

In [ ]:
sx = np.array(S["flow_fig5a"]["nch"]); sp = np.array(S["flow_fig5a"]["p"])
e = np.array(R["pnch"]["edges"]); c = np.array(R["pnch"]["counts"], float); xc = 0.5 * (e[1:] + e[:-1])
ce = np.array(R["pnch"]["counts_err"], float)
NMIN = 15

def model_density(alpha, err=False):
    """model p(N_ch) with N_ch -> alpha N_ch, as a density at STAR's points, unit area above NMIN
    (err=True: also its statistical error)"""
    dens = c / c.sum() / (np.diff(e) * alpha); derr = ce / c.sum() / (np.diff(e) * alpha)
    x = xc * alpha
    f = np.interp(sx, x, dens, left=np.nan, right=0); fe = np.interp(sx, x, derr, left=np.nan, right=0)
    m = sx >= NMIN
    norm = np.trapezoid(f[m], sx[m])
    return (f / norm, fe / norm) if err else f / norm

sp_n = sp / np.trapezoid(sp[sx >= NMIN], sx[sx >= NMIN])
alphas = np.linspace(0.8, 2.0, 241)
m = (sx >= NMIN) & (sp > 0)
chi = [np.nansum((np.log(np.maximum(model_density(a)[m], 1e-12)) - np.log(sp_n[m])) ** 2) for a in alphas]
a_best = alphas[int(np.argmin(chi))]

fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
for col, a in enumerate((1.0, a_best)):
    ax, rx = axs[0, col], axs[1, col]
    md_, mde = model_density(a, err=True)
    ax.semilogy(sx, sp_n, "o", mfc="none", color="C0", ms=4, label="STAR (arXiv:2510.19645 Fig. 5a)")
    H.band(ax, sx, md_, mde, color="C3", label="model" + ("" if a == 1 else f", N_ch × {a:.2f}"))
    ax.axvline(NMIN, color="0.6", ls=":")
    ax.set_ylim(1e-7, 0.1); ax.set_ylabel("p(N_ch), unit area above 15"); ax.legend(fontsize=9)
    ax.set_title("model as is" if a == 1 else "model multiplicity scaled")
    H.points(rx, sx[m], md_[m] / sp_n[m], mde[m] / sp_n[m], fmt="k.", capsize=0)
    rx.axhline(1, color="0.5", lw=0.8); rx.set_ylim(0, 2.5); rx.set_ylabel("model / STAR")
    rx.set_xlabel("N_ch (0.2<pT<2 GeV, |η|<1.5)")
plt.tight_layout(); plt.show()
print(f"best multiplicity scale alpha = {a_best:.2f}")

## 2. $\langle N_\mathrm{ch}\rangle$ against centrality
Classes as STAR's points (1 % wide in 0–10 %, then 10–15, 15–20 and 10 % wide); the model's centrality from
percentiles of its own $N_\mathrm{ch}$. The model's mean $N_\mathrm{part}$ and $N_\mathrm{coll}$ per class are listed for
reference.

In [ ]:
cl = R["nch_vs_cent"]["classes"]; cen = np.array([0.5 * (a + b) for a, b in cl])
mn = np.array(R["nch_vs_cent"]["nch"]); mne = np.array(R["nch_vs_cent"]["nch_err"])
sc = np.array(S["flow_fig5b"]["cent"]); sn = np.array(S["flow_fig5b"]["nch"])
fig, (ax, rx) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
ax.semilogy(sc, sn, "o", mfc="none", color="C0", label="STAR (arXiv:2510.19645 Fig. 5b)")
H.points(ax, cen, mn, mne, fmt="s", color="C3", ms=5, label="model"); ax.set_yscale("log")
ax.set_ylabel("⟨N_ch⟩ (0.2<pT<2 GeV, |η|<1.5)"); ax.legend(); ax.invert_xaxis()
r = np.interp(cen, sc, sn)
H.points(rx, cen, mn / r, mne / r, fmt="ks", ms=5); rx.axhline(1, color="0.5", lw=0.8); rx.set_ylim(0, 1.3)
rx.set_ylabel("model / STAR"); rx.set_xlabel("centrality [%]")
plt.tight_layout(); plt.show()
print(" class      STAR   model         ratio         <Npart> <Ncoll>")
for (a, b), x, y, ye, npa, nco in zip(cl, r, mn, mne, R["nch_vs_cent"]["npart"], R["nch_vs_cent"]["ncoll"]):
    print(f"{a:3d}-{b:<3d}%  {x:6.1f} {y:6.1f}±{ye:4.2f}  {y / x:5.3f}±{ye / x:6.4f}  {npa:6.1f}  {nco:6.1f}")

## 3. The EPD multiplicity (event activity)
Left: STAR's minimum-bias nMIP distribution against the model's EPD charged multiplicity × $k$ (best fit; the
response width $s$ that fits best), both normalised to the same area above nMIP 78 (the fitted range; the two
lowest bins hold the low-activity events and STAR's trigger correction and are compared separately: the fraction
of events below 78). Right: $k$ needed at each of STAR's class edges (STAR's nMIP edge over the
model's $N_\mathrm{ch}$ at the same percentile); flat means the shape is right, a rise toward peripheral classes
means the model has too few particles in low-activity events relative to central ones.

In [ ]:
mbS = S["ea_fig1"]["mb"]; lo, hi, y = (np.array(mbS[k]) for k in ("lo", "hi", "y"))
xc = 0.5 * (lo + hi)
E = R["epd"]
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
ax = axs[0]
ax.errorbar(xc, y, xerr=0.5 * (hi - lo), fmt="o", mfc="none", color="k", label="STAR MB (arXiv:2604.13935 Fig. 1)")
H.stairs(ax, E["mb_model"], E["mb_model_err"], np.append(lo, hi[-1]), color="C3",
         label=f"model, nMIP = {E['k_best']:.2f} × N_ch,EPD (s = {E['s_best']})")
for lab, x in E["edges_star_nmip"].items():
    ax.axvline(x, color="0.7", ls="--", lw=0.8); ax.text(x + 3, 4.5e-3, lab, rotation=90, color="0.5", fontsize=8)
ax.set_xlabel("nMIP_EPD"); ax.set_ylabel("self-normalised entries"); ax.legend(fontsize=9)
ax = axs[1]
labs = list(E["k_from_edges"])
H.points(ax, range(len(labs)), [E["k_from_edges"][l] for l in labs], [E["k_from_edges_err"][l] for l in labs], fmt="ks-", ms=5)
ax.axhline(E["k_best"], color="C3", ls="--", label="k from the shape fit")
ax.set_xticks(range(len(labs)), [f"edge of {l}" for l in labs], rotation=20)
ax.set_ylabel("k = nMIP edge (STAR) / N_ch,EPD edge (model)"); ax.legend()
plt.tight_layout(); plt.show()
print(f"fraction of minimum-bias events below nMIP 78: STAR {E['frac_below_fit_star']:.3f}, "
      f"model {E['frac_below_fit_model']:.3f}±{E['frac_below_fit_model_err']:.3f}")
print("k at STAR's class edges: " + ", ".join(f"{l} {E['k_from_edges'][l]:.3f}±{E['k_from_edges_err'][l]:.3f}" for l in labs))
for s, f in E["fit"].items():
    print(f"width s = {s}: k = {f['k']:.2f}, chi2 = {f['chi2']:.1f} / {f['ndf']} (3 % per bin, a shape metric)")

## 4. Model dN$_\mathrm{ch}$/dη per event-activity class
Classes from the model's EPD multiplicity (STAR's EA classes). No STAR dN/dη exists for O+O at 200 GeV; shown
for orientation (all $p_T$, charged).

In [ ]:
ee = np.array(R["eta_edges"]); ec = 0.5 * (ee[1:] + ee[:-1])
D = R["dndeta"]
fig, ax = plt.subplots(figsize=(8, 5))
for k, ((a, b), h, he) in enumerate(zip(D["classes"], D["per_class"], D["per_class_err"])):
    H.band(ax, ec, h, he, color=plt.cm.viridis(k / len(D["classes"])), label=f"{a}-{b} %")
H.band(ax, ec, D["mb"], D["mb_err"], color="k", ls="--", label="minimum bias")
for x in (2.14, 5.09):
    ax.axvspan(x, 5.09, color="0.9") if x == 2.14 else None
ax.axvspan(-5.09, -2.14, color="0.9")
ax.set_xlabel("η"); ax.set_ylabel("dN_ch/dη"); ax.legend(ncol=2, fontsize=9); ax.set_title("grey: EPD acceptance")
plt.tight_layout(); plt.show()
print(" class    <Npart>      dN/deta(|eta|<0.5)  per participant pair")
for (a, b), npa, npe, v, ve in zip(D["classes"], D["npart"], D["npart_err"], D["mid05"], D["mid05_err"]):
    pp, ppe = H.ratio(v, ve, npa / 2, npe / 2)
    print(f"{a:3d}-{b:<3d}%  {npa:6.2f}±{npe:4.2f}  {v:8.2f}±{ve:4.2f}  {float(pp):8.3f}±{float(ppe):5.3f}")

## 5. Event activity of jet-triggered events (the bias in data)
STAR's Fig. 4 shows the minimum-bias and the 7–10 / 10–30 GeV trigger distributions divided by the 5–7 GeV
trigger distribution: triggered events sit at higher activity, and in 0–60 % the ratio between trigger bins is
flat — the bias does not depend on the trigger $p_T$. The model's counterpart: jet events (production settings,
$N_\mathrm{coll}$ weighted) with a leading charged hadron above 5 GeV at $|\eta|<1.5$, per $\hat p_T$ window, and the
minimum-bias distribution divided by them. The windows leave gaps (11–18, 23–30 GeV), so the model has no
inclusive trigger sample; the per-window distributions stand in for STAR's trigger bins. The model's ratios are
taken to the 18–23 GeV window (≈ 8k triggered events): the 9–11 GeV window has only ≈ 1.4k events with a leading
hadron above 5 GeV, and its statistical noise would dominate every ratio.

In [ ]:
if "jets" not in R:
    print("jet events not yet available")
else:
    Jt = R["jets"]
    mbS = S["ea_fig1"]; lo, hi = np.array(mbS["mb"]["lo"]), np.array(mbS["mb"]["hi"]); xc = 0.5 * (lo + hi)
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    ax = axs[0]
    for n, mk in (("mb", "o"), ("trig_5_7", "s"), ("trig_10_30", "x")):
        ax.plot(xc, mbS[n]["y"], mk, mfc="none", label=f"STAR {n}")
    edges = np.append(lo, hi[-1])
    H.stairs(ax, R["epd"]["mb_model_full"], R["epd"]["mb_model_full_err"], edges, color="k", label="model MB")
    for k, (w, v) in enumerate(Jt["per_window"].items()):
        if v["n"] > 200:
            H.stairs(ax, v["h"], v["h_err"], edges, color=f"C{k + 1}", label=f"model jets p̂T {w} (n={v['n']})")
    ax.set_xlabel("nMIP_EPD"); ax.set_ylabel("self-normalised entries"); ax.legend(fontsize=8)
    ax = axs[1]
    s57 = np.array(mbS["trig_5_7"]["y"])
    r4 = S["ea_fig4"]
    for n, mk in (("mb", "o"), ("trig_10_30", "x")):
        rlo = np.array(r4[n]["lo"]); ax.plot(0.5 * (rlo + np.array(r4[n]["hi"])), r4[n]["y"], mk, mfc="none",
                                              label=f"STAR {n} / 5-7 GeV (Fig. 4)")
    REF = "18-23"   # the reference window (STAR's is its lowest trigger bin, 5-7 GeV); 9-11 has too few triggered events
    ref, refe = Jt["per_window"][REF]["h"], Jt["per_window"][REF]["h_err"]
    H.band(ax, xc, *H.ratio(R["epd"]["mb_model_full"], R["epd"]["mb_model_full_err"], ref, refe), color="k", label=f"model MB / jets {REF}")
    for k, (w, v) in enumerate(Jt["per_window"].items()):
        if w != REF and v["n"] > 200:
            H.band(ax, xc, *H.ratio(v["h"], v["h_err"], ref, refe), color=f"C{k + 1}", label=f"model jets {w} / jets {REF}")
    for lab, x in mbS["class_edges"].items():
        ax.axvline(x, color="0.7", ls="--", lw=0.8)
    ax.axhline(1, color="0.5", lw=0.8); ax.set_ylim(0.3, 2.0); ax.set_xlabel("nMIP_EPD"); ax.set_ylabel("ratio"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Results and what they mean
2026-11-23: 49,996 minimum-bias and 50,000 jet events (production settings, jet in vacuum); statistical errors
throughout. Model results, not vetted physics results.

1. **The multiplicity is too low, and more so the more peripheral.** $\langle N_\mathrm{ch}\rangle$ model/STAR is 0.85
   (0–1 %), 0.79 (9–10 %), 0.71 (20–30 %), 0.55 (50–60 %), 0.24 (70–80 %); the statistical errors are below 0.5 %, the
   deficit is physics. The shape of p($N_\mathrm{ch}$) above 15 is matched only after scaling the model's $N_\mathrm{ch}$ by
   $\alpha$ = 1.23. The peripheral ratios rest on STAR's Glauber fit at low $N_\mathrm{ch}$ (it adds ≈ 3 % of events);
   the central deficit does not.
2. **The rise with $N_\mathrm{part}$ is too steep.** Mid-rapidity dN/dη per participant pair falls from 2.90 ± 0.01
   (0–10 % EA) to 1.36 ± 0.02 (80–100 %), against ≈ 2.3–2.5 in p+p: a single nucleon–nucleon collision makes too few
   particles. The EPD says the same: $k$ at STAR's class edges rises from 2.754 ± 0.011 (0–10 %) to 3.10 ± 0.02
   (60–80 %), and the model has 46.0 ± 0.2 % of its events below nMIP 78 against STAR's 41.8 %. The shape above 78
   fits with one $k$ = 2.60 (s = 1.0).
3. **Event activity of jet events (STAR Fig. 4).** Relative to the 18–23 GeV window the higher windows are flat above
   nMIP 78 (30–40: 0.99 ± 0.02, 40–55: 0.99 ± 0.02), as STAR's 10–30/5–7 GeV in 0–60 %; in the lowest bin (nMIP < 39)
   they rise with the hard scale (1.28 ± 0.04, 1.35 ± 0.05; STAR 10–30/5–7: 1.52): harder triggers pull events to very
   low forward activity, in the model through the hard-momentum subtraction. Minimum bias over jet events falls
   faster than in STAR (0.29–0.33 against 0.44–0.51 above nMIP 300): the model's minimum-bias distribution is too
   steep at high activity, as in 1 and 2.

**What it means.** The bulk set (the MAP of 2601.17234, calibrated on Au+Au and central d+Au) underpredicts O+O, most
of all where few nucleons collide. The working hypothesis (tuning page): the MAP's rapidity loss gives a too low
yield per collision and no shadowing, so the rise with $N_\mathrm{part}$ is too steep; the bulk formation time (0.5
against 0.424 fm/c) is a ≈ 4 % effect. Both feed the centrality bias: the peripheral classes, where the bias is
largest, are where the bulk is worst.